# Galaxy Rings – Downloader & CSV Builder
Este notebook estandariza múltiples CSVs de galaxias (con distintos esquemas), limpia nulos/outliers, descarga imágenes FITS desde Legacy Survey y genera CSVs listos para entrenar.

- Variante: **SEPARADO (un CSV por dataset)**


In [1]:
# Imports
import os
from pathlib import Path
import re
import time
import urllib.request
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder


In [2]:
# =========================
# 0) Configuración de rutas
# =========================
# Estructura recomendada:
# PROJECT_ROOT/
#   Data/        <- CSVs de entrada y salidas
#   Images/      <- descarga de  FITS


PROJECT_ROOT = Path(os.getcwd()).resolve()
DATA_DIR   = PROJECT_ROOT / "Data"
IMAGE_DIR  = PROJECT_ROOT / "Images"

DATA_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR     :", DATA_DIR)
print("IMAGE_DIR    :", IMAGE_DIR)


PROJECT_ROOT: C:\Users\dark_\Downloads\Project_2 - Copy
DATA_DIR     : C:\Users\dark_\Downloads\Project_2 - Copy\Data
IMAGE_DIR    : C:\Users\dark_\Downloads\Project_2 - Copy\Images


In [3]:
# ======================================================
# 1) Utilidades: limpieza, estandarización y etiquetado
# ======================================================

def coerce_numeric(df: pd.DataFrame, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def clean_basic(df: pd.DataFrame, required_cols, drop_duplicates_by=None):
    """Limpieza rápida:
    - normaliza nombres de columnas (strip)
    - fuerza NaN en blancos
    - elimina filas con NaN en columnas requeridas
    - elimina duplicados (opcional)
    """
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    # Convertir strings vacíos / espacios a NaN
    df = df.replace(r"^\s*$", np.nan, regex=True)

    missing_before = df[required_cols].isna().sum() if all(c in df.columns for c in required_cols) else None

    df = df.dropna(subset=[c for c in required_cols if c in df.columns]).copy()

    if drop_duplicates_by is not None and drop_duplicates_by in df.columns:
        df = df.drop_duplicates(subset=[drop_duplicates_by]).copy()

    missing_after = df[required_cols].isna().sum() if all(c in df.columns for c in required_cols) else None

    return df, missing_before, missing_after

def z_sigma_clip(df: pd.DataFrame, z_col="z", sigma=2.0):
    if z_col not in df.columns:
        return df
    z = pd.to_numeric(df[z_col], errors="coerce")
    mu, sd = z.mean(), z.std()
    if pd.isna(mu) or pd.isna(sd) or sd == 0:
        return df
    lo, hi = mu - sigma*sd, mu + sigma*sd
    return df[(z >= lo) & (z <= hi)].copy()

# ---- Etiquetas de anillos: 0, 4, 8, 12 (se descarta 2=nuclear y 16=pseudo si aparecen)
def to_4class(code):
    if pd.isna(code):
        return np.nan
    code = int(round(float(code)))
    if code == 0:   # no ring
        return 0
    if code == 4:   # inner ring
        return 1
    if code == 8:   # outer ring
        return 2
    if code == 12:  # inner + outer
        return 3
    # nuclear(2) y pseudo(16) se descartan por decisión de dominio
    return np.nan

def add_ring_targets(df: pd.DataFrame, anillos_col="anillos"):
    df = df.copy()
    df[anillos_col] = pd.to_numeric(df[anillos_col], errors="coerce")
    df = df.dropna(subset=[anillos_col]).copy()
    df[anillos_col] = df[anillos_col].round().astype(int)

    # descartar nuclear y pseudo si aparecen
    df = df[(df[anillos_col] != 2) & (df[anillos_col] != 16)].copy()

    df["ring_class"] = df[anillos_col].apply(to_4class)
    df = df.dropna(subset=["ring_class"]).copy()
    df["ring_class"] = df["ring_class"].astype(int)

    # One-hot: ring_class_0..3
    enc = OneHotEncoder(sparse_output=False)
    one_hot = enc.fit_transform(df[["ring_class"]])
    one_hot_df = pd.DataFrame(one_hot, columns=enc.get_feature_names_out(["ring_class"]), index=df.index)
    df = pd.concat([df, one_hot_df], axis=1)

    # Multilabel explícito (útil si entrenas inner/outer por separado)
    df["inner_ring"] = (df[anillos_col].isin([4, 12])).astype(int)
    df["outer_ring"] = (df[anillos_col].isin([8, 12])).astype(int)

    return df

def standardize_schema(df: pd.DataFrame, schema: str):
    """Devuelve df con columnas estándar: id_str, ra, dec, z, anillos"""
    df = df.copy()
    if schema == "sdss_original":
        # objID ra dec z anillos
        rename = {"objID":"id_str"}
        df = df.rename(columns=rename)
        df["id_str"] = df["id_str"].astype(str)
        df = coerce_numeric(df, ["ra","dec","z","anillos"])
        return df[["id_str","ra","dec","z","anillos"]].copy()

    if schema == "manga_alt":
        # name anillos objra objdec nsa_z
        rename = {"name":"id_str","objra":"ra","objdec":"dec","nsa_z":"z"}
        df = df.rename(columns=rename)
        df["id_str"] = df["id_str"].astype(str)
        df = coerce_numeric(df, ["ra","dec","z","anillos"])
        return df[["id_str","ra","dec","z","anillos"]].copy()

    if schema == "csrg_catalog":
        # Name RA2000 Dec-00 ... + ring info in Class/Variety
        rename = {"Name":"id_str","RA2000":"ra","Dec-00":"dec"}
        df = df.rename(columns=rename)
        df["id_str"] = df["id_str"].astype(str).str.strip()
        df = coerce_numeric(df, ["ra","dec"])

        # ---- infer anillos from morphology strings (heurístico)
        # outer ring: Class contiene 'R' (R, R?, RL) al inicio
        cls = df.get("Class", pd.Series([""]*len(df), index=df.index)).fillna("").astype(str).str.strip()
        var = df.get("Variety", pd.Series([""]*len(df), index=df.index)).fillna("").astype(str).str.strip()

        outer = cls.str.contains(r"^R", regex=True)  # R, R?, RL
        # inner ring: Variety contiene 'r' o 'rs' (pero no vacío). 'rs' se interpreta como transición ring/spiral.
        inner = var.str.contains(r"r", regex=True)

        df["anillos"] = (inner.astype(int)*4 + outer.astype(int)*8).astype(int)
        # z no existe en este catálogo: rellena NaN
        df["z"] = np.nan
        return df[["id_str","ra","dec","z","anillos"]].copy()

    raise ValueError(f"Schema desconocido: {schema}")

def quick_profile(df: pd.DataFrame, name="df"):
    print(f"\n[{name}] shape:", df.shape)
    print(df.head(3))
    print("\nNA por columna (top 10):")
    print(df.isna().sum().sort_values(ascending=False).head(10))
    return df


In [46]:
# ==========================================
# 2) Cargar CSVs y estandarizarlos a schema UNIFICADO
# ==========================================


CSV_ORIGINAL = r"C:\Users\dark_\Downloads\Project_2\Data\dataset.csv"
CSV_MANGA    = r"C:\Users\dark_\Downloads\Project_2\Data\MaNGA_rings.csv"
CSV_CSRG     = r"C:\Users\dark_\Downloads\Project_2\Data\CSRG_Buta.csv"

In [47]:
# ==========================================
# Exploración rápida del dataset CSRG
# ==========================================

raw = pd.read_csv(CSV_CSRG)

# limpiar nombres de columnas
raw.columns = raw.columns.str.strip()

print("Columnas del dataset:")
print(raw.columns.tolist())

print("\nShape del dataset:")
print(raw.shape)

# revisar columnas morfológicas importantes
cols_to_check = ["Class", "Family", "Variety", "TType"]

for col in cols_to_check:
    if col in raw.columns:
        print(f"\n==== Valores únicos en {col} ====")
        print(raw[col].value_counts(dropna=False).head(30))

# revisar también RA/DEC por si acaso
for col in ["RA2000", "Dec-00"]:
    if col in raw.columns:
        print(f"\nStats {col}")
        print(raw[col].describe())

# ver algunos ejemplos
print("\nPrimeras filas:")
display(raw.head(10))

Columnas del dataset:
['Name', 'RA2000', 'DEC2000', 'Class', 'Family', 'Variety', 'TType', 'aOut', 'bOut', 'PAOut', 'aIn', 'bIn', 'PAIn']

Shape del dataset:
(3770, 13)

==== Valores únicos en Class ====
Class
NaN     1594
RP       514
R        439
R2P      350
R1P      328
R1       215
RP:       65
RL        62
R:        36
L         32
R?        26
RP?       16
Rng       13
PR?       11
R2P:      10
R1P:      10
R2P?       9
Rng?       9
R2         7
RPL        7
R1L        3
R1?        2
R2PL       2
RL?        2
L:         2
R1P?       1
RLm        1
RPR?       1
PR         1
L?         1
Name: count, dtype: int64

==== Valores únicos en Family ====
Family
SB      1863
SA       660
SX       516
SX-      238
SX+      209
S         86
SB:       49
SX:       46
SA:       38
NaN       23
SB?       12
SX-:       8
SA?        8
SX?        6
SX+:       6
SX-?       1
I          1
Name: count, dtype: int64

==== Valores únicos en Variety ====
Variety
r       1318
s        699
rs       507


,Name,RA2000,DEC2000,Class,Family,Variety,TType,aOut,bOut,PAOut,aIn,bIn,PAIn
0,CSRG 1,0.640265,-25.198298,R?,SA,r,1.0,0.68,0.51,0.0,0.32,0.25,0.0
1,CSRG 2,0.677001,-53.748590,RL,SB,NaN,0.0,0.51,0.40,87.6,0.00,0.00,0.0
2,CSRG 3,0.663653,-32.226979,R,SB,l,-2.0,0.50,0.40,84.4,0.26,0.20,0.0
3,NGC 7812,0.728841,-34.236029,NaN,SX,rs,2.0,0.00,0.00,0.0,0.57,0.39,0.0
4,CSRG 4,0.696056,-54.247283,NaN,SB,rl,0.0,0.00,0.00,0.0,0.40,0.27,2.3
5,ESO 293-31,1.592377,-42.180768,NaN,SX-,rs-,2.0,0.00,0.00,0.0,0.39,0.20,0.0
6,ESO 149-15,0.848658,-53.336724,NaN,SX,r,5.0,0.00,0.00,0.0,0.39,0.29,3.9
7,IC 5382,0.899068,-65.203648,NaN,SB,rs,3.0,0.00,0.00,0.0,0.40,0.23,33.0
8,CSRG 5,0.964373,-43.632139,NaN,SB,r,1.5,0.00,0.00,0.0,0.51,0.40,16.2
9,CSRG 6,0.889707,-44.547241,R1,SB,s,-1.5,0.59,0.47,85.3,0.00,0.00,0.0


### Inferencia de la variable **anillos** en el catálogo CSRG

El catálogo **CSRG (Catalog of Southern Ringed Galaxies)** no proporciona directamente una variable numérica equivalente a la columna **`anillos`** utilizada en los otros datasets del proyecto. En su lugar, describe la morfología de las galaxias mediante etiquetas textuales en las columnas **`Class`** y **`Variety`**, que siguen la nomenclatura morfológica clásica utilizada en estudios de galaxias con anillos.

Para poder integrar este catálogo con los otros conjuntos de datos, fue necesario **inferir la presencia de anillos internos y externos** a partir de dichas etiquetas morfológicas y posteriormente codificar esta información utilizando el mismo esquema de clases empleado en el dataset original.

#### Interpretación de las columnas morfológicas

**1. Columna `Class`**

Esta columna describe la morfología global relacionada con estructuras de anillo externas. En la clasificación de Buta utilizada en CSRG, los valores que comienzan con la letra **R** representan distintos tipos de anillos externos o pseudo-anillos. Algunos ejemplos observados en el dataset incluyen:

- `R` → outer ring  
- `R1`, `R2` → variantes de outer ring  
- `RP`, `R1P`, `R2P` → pseudo-rings  
- `RL` → ring-lens  
- `R?` → anillo externo incierto  

Debido a esta convención, se adoptó el siguiente criterio:

> **Si `Class` comienza con la letra `R`, la galaxia se considera que posee un anillo externo.**

---

**2. Columna `Variety`**

La columna `Variety` describe estructuras internas relacionadas con la presencia de anillos o transiciones entre anillos y brazos espirales. Los valores más comunes son:

- `r` → inner ring  
- `rs` → transición entre ring y spiral  
- `s` → estructura espiral sin anillo

Dado que tanto `r` como `rs` implican la presencia de una estructura de anillo interno o relacionada con él, se utilizó el siguiente criterio:

> **Si `Variety` contiene la letra `r`, la galaxia se considera que posee un anillo interno.**

---

#### Codificación final de la variable `anillos`

Una vez identificada la presencia de **anillo interno** y **anillo externo**, se generó la variable `anillos` utilizando el mismo esquema de codificación del dataset original:

| Valor | Interpretación |
|------|----------------|
0 | sin anillos |
4 | anillo interno |
8 | anillo externo |
12 | anillo interno y externo |

La codificación se calculó mediante la siguiente expresión: anillos = 4 * inner_ring + 8 * outer_ring


donde:

- `inner_ring = 1` si `Variety` contiene `"r"`
- `outer_ring = 1` si `Class` comienza con `"R"`

---

#### Justificación

Este procedimiento permite **integrar el catálogo CSRG con los otros datasets del proyecto manteniendo un esquema de clases consistente**, al mismo tiempo que respeta la interpretación morfológica establecida en la literatura astronómica para galaxias con anillos.


In [48]:
# ==========================================
# 2) Cargar CSVs y estandarizarlos a schema
# ==========================================


def load_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace("\u200b", "", regex=False)  # zero-width
        .str.replace("\ufeff", "", regex=False)  # BOM
    )
    return df

def ensure_cols(df: pd.DataFrame, cols: list[str]) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"Faltan columnas: {missing}\nColumnas disponibles: {list(df.columns)}")

def infer_anillos_csrg(df: pd.DataFrame) -> pd.DataFrame:
    """
    outer_ring: Class empieza con 'R'
    inner_ring: Variety contiene 'r' (r o rs)
    anillos = 4*inner + 8*outer  -> {0,4,8,12}
    """
    df = df.copy()

    # Asegurar columnas morfológicas
    if "Class" not in df.columns:
        df["Class"] = ""
    if "Variety" not in df.columns:
        df["Variety"] = ""

    cls = df["Class"].fillna("").astype(str).str.strip().str.upper()
    var = df["Variety"].fillna("").astype(str).str.strip().str.lower()

    df["outer_ring"] = cls.str.startswith("R").astype("int64")
    df["inner_ring"] = var.str.contains("r", regex=False).astype("int64")
    df["anillos"] = (df["inner_ring"] * 4 + df["outer_ring"] * 8).astype("int64")
    return df

datasets = []

# ---------- ORIGINAL ----------
if CSV_ORIGINAL:
    raw = load_csv(Path(CSV_ORIGINAL))
    quick_profile(raw, "raw_original")

    #  ORIGINAL ya trae: objID, ra, dec, z, anillos
    std, _, _ = clean_basic(raw, required_cols=["objID","ra","dec","z","anillos"], drop_duplicates_by="objID")
    std = standardize_schema(std, "sdss_original")
    std["dataset"] = "original"
    datasets.append(std)

# ---------- MANGA ----------
if CSV_MANGA:
    raw = load_csv(Path(CSV_MANGA))
    quick_profile(raw, "raw_manga")

    # Renombrar a esquema estándar ANTES de clean_basic
    rename_map = {
        "name": "objID",
        "objra": "ra",
        "objdec": "dec",
        "nsa_z": "z",
        "anillos": "anillos",
    }
    # mapa case-insensitive
    lower_to_real = {c.lower(): c for c in raw.columns}
    for k, v in rename_map.items():
        if k in lower_to_real:
            raw = raw.rename(columns={lower_to_real[k]: v})

    std, _, _ = clean_basic(raw, required_cols=["objID","ra","dec","z","anillos"], drop_duplicates_by="objID")
    std = standardize_schema(std, "sdss_original")  
    std["dataset"] = "manga"
    datasets.append(std)

# ---------- CSRG ----------
if CSV_CSRG:
    raw = load_csv(Path(CSV_CSRG))
    quick_profile(raw, "raw_csrg")

    # Renombrar a esquema base
    raw = raw.rename(columns={"Name": "objID", "RA2000": "ra"})
    if "DEC2000" in raw.columns:
        raw = raw.rename(columns={"DEC2000": "dec"})
    elif "Dec-00" in raw.columns:
        raw = raw.rename(columns={"Dec-00": "dec"})
    else:
        # fallback: buscar algo que contenga 'dec' y '2000'
        for c in raw.columns:
            cl = c.lower().replace(" ", "")
            if "dec" in cl and ("2000" in cl or "j2000" in cl or "00" in cl):
                raw = raw.rename(columns={c: "dec"})
                break

    # Imputar anillos desde morfología
    raw = infer_anillos_csrg(raw)

    # Limpieza SIN z (porque CSRG no lo tiene)
    std, _, _ = clean_basic(raw, required_cols=["objID","ra","dec","anillos"], drop_duplicates_by="objID")

    # Ahora agregamos z como NaN (para compatibilidad del schema)
    std["z"] = np.nan

    # Si standardize_schema espera z presente, ya lo tiene
    std = standardize_schema(std, "sdss_original")
    std["dataset"] = "csrg"
    datasets.append(std)

if not datasets:
    raise RuntimeError("No cargaste ningún CSV. Define CSV_ORIGINAL / CSV_MANGA / CSV_CSRG con rutas válidas.")

df_all = pd.concat(datasets, ignore_index=True)
#quick_profile(df_all, "df_all (estandarizado)")
#display(df_all.head())


[raw_original] shape: (8528, 5)
                 objID         ra       dec         z  anillos
0  1237648721210769659  134.44717 -0.199973  0.028206      0.0
1  1237648705657307354  198.23356  0.941188  0.048037      0.0
2  1237648705120895059  199.29492  0.527571  0.024114      0.0

NA por columna (top 10):
objID      0
ra         0
dec        0
z          0
anillos    0
dtype: int64

[raw_manga] shape: (680, 5)
                name  anillos       objra     objdec     nsa_z
0   manga-10001-1902        4  134.193923  56.786747  0.025391
1   manga-10001-6103        8  134.008123  57.390964  0.040626
2  manga-10213-12705        4  122.931168  41.554978  0.031548

NA por columna (top 10):
name       0
anillos    0
objra      0
objdec     0
nsa_z      0
dtype: int64

[raw_csrg] shape: (3770, 13)
           Name    RA2000    DEC2000 Class Family Variety  TType  aOut  bOut  \
0  CSRG 1        0.640265 -25.198298  R?     SA      r       1.0  0.68  0.51   
1  CSRG 2        0.677001 -53.748590

In [49]:
print("\nResumen por dataset antes de concatenar:\n")

for i, df in enumerate(datasets):
    print(f"Dataset index: {i}")

    if df.empty:
        print("Dataset vacío")
        print("-"*50)
        continue

    # nombre del dataset
    if "dataset" in df.columns:
        name = df["dataset"].iloc[0]
    else:
        name = f"dataset_{i}"

    print(f"Nombre: {name}")
    print("Shape:", df.shape)

    if "anillos" in df.columns:
        print("\nDistribución de anillos:")
        print(df["anillos"].value_counts())

    print("\nHead:")
    display(df.head())

    print("-"*50)

# ==========================================
# Concatenar datasets
# ==========================================

df_all = pd.concat(datasets, ignore_index=True)

print("\nDataset combinado:")
print("Shape total:", df_all.shape)

print("\nDistribución global de anillos:")
print(df_all["anillos"].value_counts())

print("\nConteo por dataset:")
print(df_all["dataset"].value_counts())

quick_profile(df_all, "df_all (estandarizado)")
display(df_all.head())


Resumen por dataset antes de concatenar:

Dataset index: 0
Nombre: original
Shape: (8527, 6)

Distribución de anillos:
anillos
0.0     6659
4.0      857
12.0     372
16.0     342
8.0      186
2.0      111
Name: count, dtype: int64

Head:


,id_str,ra,dec,z,anillos,dataset
0,1237648721210769659,134.44717,-0.199973,0.028206,0.0,original
1,1237648705657307354,198.23356,0.941188,0.048037,0.0,original
2,1237648705120895059,199.29492,0.527571,0.024114,0.0,original
3,1237648720150724863,165.74061,-0.962095,0.033483,0.0,original
4,1237649919509594232,31.37202,13.251016,0.024694,0.0,original


--------------------------------------------------
Dataset index: 1
Nombre: manga
Shape: (680, 6)

Distribución de anillos:
anillos
4     459
8     161
12     60
Name: count, dtype: int64

Head:


,id_str,ra,dec,z,anillos,dataset
0,manga-10001-1902,134.193923,56.786747,0.025391,4,manga
1,manga-10001-6103,134.008123,57.390964,0.040626,8,manga
2,manga-10213-12705,122.931168,41.554978,0.031548,4,manga
3,manga-10213-1901,124.082784,43.447637,0.024198,4,manga
4,manga-10213-6103,121.096387,43.582589,0.040901,4,manga


--------------------------------------------------
Dataset index: 2
Nombre: csrg
Shape: (3692, 6)

Distribución de anillos:
anillos
4     1567
8     1087
12     964
0       74
Name: count, dtype: int64

Head:


,id_str,ra,dec,z,anillos,dataset
0,CSRG 1,0.640265,-25.198298,NaN,12,csrg
1,CSRG 2,0.677001,-53.748590,NaN,8,csrg
2,CSRG 3,0.663653,-32.226979,NaN,8,csrg
3,NGC 7812,0.728841,-34.236029,NaN,4,csrg
4,CSRG 4,0.696056,-54.247283,NaN,4,csrg


--------------------------------------------------

Dataset combinado:
Shape total: (12899, 6)

Distribución global de anillos:
anillos
0.0     6733
4.0     2883
8.0     1434
12.0    1396
16.0     342
2.0      111
Name: count, dtype: int64

Conteo por dataset:
dataset
original    8527
csrg        3692
manga        680
Name: count, dtype: int64

[df_all (estandarizado)] shape: (12899, 6)
                id_str         ra       dec         z  anillos   dataset
0  1237648721210769659  134.44717 -0.199973  0.028206      0.0  original
1  1237648705657307354  198.23356  0.941188  0.048037      0.0  original
2  1237648705120895059  199.29492  0.527571  0.024114      0.0  original

NA por columna (top 10):
z          3692
id_str        0
ra            0
dec           0
anillos       0
dataset       0
dtype: int64


,id_str,ra,dec,z,anillos,dataset
0,1237648721210769659,134.44717,-0.199973,0.028206,0.0,original
1,1237648705657307354,198.23356,0.941188,0.048037,0.0,original
2,1237648705120895059,199.29492,0.527571,0.024114,0.0,original
3,1237648720150724863,165.74061,-0.962095,0.033483,0.0,original
4,1237649919509594232,31.37202,13.251016,0.024694,0.0,original


In [50]:
# ==========================================
# Asegurar tipos correctos
# ==========================================

df_all["anillos"] = pd.to_numeric(df_all["anillos"], errors="coerce").fillna(0).astype("int64")

# también conviene asegurar tipos de coordenadas
df_all["ra"] = pd.to_numeric(df_all["ra"], errors="coerce")
df_all["dec"] = pd.to_numeric(df_all["dec"], errors="coerce")
df_all["z"] = pd.to_numeric(df_all["z"], errors="coerce")

print("Tipos de columnas:")
print(df_all.dtypes)

display(df_all.head())

print("Valores únicos de anillos:")
print(sorted(df_all["anillos"].unique()))

Tipos de columnas:
id_str         str
ra         float64
dec        float64
z          float64
anillos      int64
dataset        str
dtype: object


,id_str,ra,dec,z,anillos,dataset
0,1237648721210769659,134.44717,-0.199973,0.028206,0,original
1,1237648705657307354,198.23356,0.941188,0.048037,0,original
2,1237648705120895059,199.29492,0.527571,0.024114,0,original
3,1237648720150724863,165.74061,-0.962095,0.033483,0,original
4,1237649919509594232,31.37202,13.251016,0.024694,0,original


Valores únicos de anillos:
[np.int64(0), np.int64(2), np.int64(4), np.int64(8), np.int64(12), np.int64(16)]


In [52]:
# =====================================
# 3) Limpieza adicional (sin sigma-clip en z)
# =====================================

SIGMA_Z = 0   # desactivado porque CSRG no tiene z

df_all = df_all.copy()

# asegurar tipos numéricos
df_all = coerce_numeric(df_all, ["ra", "dec", "z", "anillos"])

# convertir anillos a entero
df_all["anillos"] = pd.to_numeric(df_all["anillos"], errors="coerce").astype("Int64")

# mantener clases válidas
df_all = df_all[df_all["anillos"].isin([0,2,4,8,12,16])].copy()

# generar targets para el modelo
df_all = add_ring_targets(df_all, anillos_col="anillos")

quick_profile(df_all, "df_all (limpio + labels)")

print("\nDistribución anillos:")
print(df_all["anillos"].value_counts().sort_index())

print("\nDistribución por dataset:")
print(pd.crosstab(df_all["dataset"], df_all["anillos"]))


[df_all (limpio + labels)] shape: (12446, 13)
                id_str         ra       dec         z  anillos   dataset  \
0  1237648721210769659  134.44717 -0.199973  0.028206        0  original   
1  1237648705657307354  198.23356  0.941188  0.048037        0  original   
2  1237648705120895059  199.29492  0.527571  0.024114        0  original   

   ring_class  ring_class_0  ring_class_1  ring_class_2  ring_class_3  \
0           0           1.0           0.0           0.0           0.0   
1           0           1.0           0.0           0.0           0.0   
2           0           1.0           0.0           0.0           0.0   

   inner_ring  outer_ring  
0           0           0  
1           0           0  
2           0           0  

NA por columna (top 10):
z               3692
id_str             0
ra                 0
dec                0
anillos            0
dataset            0
ring_class         0
ring_class_0       0
ring_class_1       0
ring_class_2       0
dtype: 

In [54]:
print(df_all.shape)
print(df_all["dataset"].value_counts())
display(df_all.head())

(12446, 13)
dataset
original    8074
csrg        3692
manga        680
Name: count, dtype: int64


,id_str,ra,dec,z,anillos,dataset,ring_class,ring_class_0,ring_class_1,ring_class_2,ring_class_3,inner_ring,outer_ring
0,1237648721210769659,134.44717,-0.199973,0.028206,0,original,0,1.0,0.0,0.0,0.0,0,0
1,1237648705657307354,198.23356,0.941188,0.048037,0,original,0,1.0,0.0,0.0,0.0,0,0
2,1237648705120895059,199.29492,0.527571,0.024114,0,original,0,1.0,0.0,0.0,0.0,0,0
3,1237648720150724863,165.74061,-0.962095,0.033483,0,original,0,1.0,0.0,0.0,0.0,0,0
4,1237649919509594232,31.37202,13.251016,0.024694,0,original,0,1.0,0.0,0.0,0.0,0,0


In [60]:
# ====================================================
# 4B) Downloader o
# ====================================================

import re
import time
import urllib.request
import urllib.parse
import urllib.error
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

def sanitize_id(x: str) -> str:
    x = str(x).strip()
    x = re.sub(r"[^A-Za-z0-9_\-]+", "_", x)
    return x

def build_legacy_url(ra, dec, size=256, pixscale=0.262, layer="ls-dr9", bands="grz", endpoint="cutout.fits"):
    base = f"https://www.legacysurvey.org/viewer/{endpoint}?"
    params = {
        "ra": float(ra),
        "dec": float(dec),
        "layer": layer,
        "pixscale": float(pixscale),
        "size": int(size),
        "bands": bands,
    }
    return base + urllib.parse.urlencode(params)

def valid_radec(ra, dec) -> bool:
    try:
        ra = float(ra)
        dec = float(dec)
    except Exception:
        return False
    return (0.0 <= ra < 360.0) and (-90.0 <= dec <= 90.0)

def download_legacy_fits_fast(ra, dec, out_path: Path, size=256, pixscale=0.262, layer="ls-dr9", bands="grz"):
    out_path.parent.mkdir(parents=True, exist_ok=True)

    urls = [
        build_legacy_url(ra, dec, size=size, pixscale=pixscale, layer=layer, bands=bands, endpoint="cutout.fits"),
        build_legacy_url(ra, dec, size=size, pixscale=pixscale, layer=layer, bands=bands, endpoint="fits-cutout"),
    ]

    last_err = None
    for url in urls:
        try:
            urllib.request.urlretrieve(url, out_path)
            if out_path.exists() and out_path.stat().st_size > 0:
                return out_path
        except Exception as e:
            last_err = e

    raise RuntimeError(f"No se pudo descargar FITS ra={ra}, dec={dec}. Último error: {last_err}")

def ensure_images_fast(df: pd.DataFrame, image_dir: Path, sleep_s=0.05, overwrite=False,
                       size=256, pixscale=0.262, layer="ls-dr9", bands="grz"):
    df = df.copy()
    image_dir.mkdir(parents=True, exist_ok=True)

    paths = []
    failed_rows = []
    n_total = len(df)
    n_dl = 0
    n_fail = 0

    for row in tqdm(df.itertuples(index=False), total=n_total, desc="Descargando FITS"):
        rid = sanitize_id(getattr(row, "id_str"))
        ra = getattr(row, "ra")
        dec = getattr(row, "dec")

        fname = f"{rid}_0.fits"
        outp = image_dir / fname

        if outp.exists() and (not overwrite) and outp.stat().st_size > 0:
            paths.append(str(outp))
            continue

        if not valid_radec(ra, dec):
            paths.append(np.nan)
            failed_rows.append({"id_str": rid, "ra": ra, "dec": dec, "reason": "invalid_radec"})
            n_fail += 1
            continue

        try:
            download_legacy_fits_fast(ra, dec, outp, size=size, pixscale=pixscale, layer=layer, bands=bands)
            n_dl += 1
            paths.append(str(outp))
            if sleep_s:
                time.sleep(sleep_s)
        except Exception as e:
            paths.append(np.nan)
            failed_rows.append({"id_str": rid, "ra": ra, "dec": dec, "reason": str(e)})
            n_fail += 1
            if n_fail <= 20 or n_fail % 100 == 0:
                print(f"[WARN] ({n_fail}) fallo id={rid} ra={ra} dec={dec} -> {e}")

    df["file_loc"] = paths
    failed_df = pd.DataFrame(failed_rows)

    print(f"Descargas nuevas: {n_dl}/{n_total} | fallos: {n_fail}")
    return df, failed_df


# Reanudar sobre dataset
df_to_dl, failed_df = ensure_images_fast(
    df_all,
    IMAGE_DIR,
    sleep_s=0.05,
    overwrite=False,
    size=256,
    pixscale=0.262,
    layer="ls-dr9",
    bands="grz"
)

print("Con imagen:", df_to_dl["file_loc"].notna().sum(), "/", len(df_to_dl))

if len(failed_df) > 0:
    failed_df.to_csv("Exports/failed_downloads.csv", index=False)
    print("Fallos guardados en Exports/failed_downloads.csv")

display(df_to_dl.head())

Descargando FITS:  18%|█▊        | 2274/12446 [00:01<00:07, 1393.42it/s]

[WARN] (1) fallo id=1237661055281725940 ra=49.593772 dec=41.409774 -> No se pudo descargar FITS ra=49.593772, dec=41.409774. Último error: HTTP Error 500: Internal Server Error
[WARN] (2) fallo id=1237661055818662378 ra=50.091912 dec=41.640915 -> No se pudo descargar FITS ra=50.091912, dec=41.640915. Último error: HTTP Error 500: Internal Server Error
[WARN] (3) fallo id=1237661055281725866 ra=49.68814 dec=41.488764 -> No se pudo descargar FITS ra=49.68814, dec=41.488764. Último error: HTTP Error 500: Internal Server Error
[WARN] (4) fallo id=1237661055281659994 ra=49.366154 dec=41.488063 -> No se pudo descargar FITS ra=49.366154, dec=41.488063. Último error: HTTP Error 500: Internal Server Error
[WARN] (5) fallo id=1237661059574006107 ra=49.141793 dec=42.025852 -> No se pudo descargar FITS ra=49.141793, dec=42.025852. Último error: HTTP Error 500: Internal Server Error
[WARN] (6) fallo id=1237661056355205202 ra=49.557465 dec=42.53797 -> No se pudo descargar FITS ra=49.557465, dec=42.5

Descargando FITS:  19%|█▊        | 2314/12446 [00:17<01:46, 95.07it/s]  

[WARN] (7) fallo id=1237660557596623183 ra=63.56076 dec=24.660914 -> No se pudo descargar FITS ra=63.56076, dec=24.660914. Último error: HTTP Error 500: Internal Server Error


Descargando FITS:  19%|█▉        | 2344/12446 [00:19<02:01, 83.21it/s]

...

Descargando FITS:  98%|█████████▊| 12188/12446 [1:22:01<02:45,  1.56it/s]  

[WARN] (1800) fallo id=IC_5252 ra=341.79320615820905 dec=-69.24995722552126 -> No se pudo descargar FITS ra=341.79320615820905, dec=-69.24995722552126. Último error: HTTP Error 500: Internal Server Error


Descargando FITS: 100%|██████████| 12446/12446 [1:24:44<00:00,  2.45it/s]

Descargas nuevas: 509/12446 | fallos: 1908
Con imagen: 10538 / 12446
Fallos guardados en Exports/failed_downloads.csv


,id_str,ra,dec,z,anillos,dataset,ring_class,ring_class_0,ring_class_1,ring_class_2,ring_class_3,inner_ring,outer_ring,file_loc
0,1237648721210769659,134.44717,-0.199973,0.028206,0,original,0,1.0,0.0,0.0,0.0,0,0,C:\Users\dark_\Downloads\Project_2\Images\1237...
1,1237648705657307354,198.23356,0.941188,0.048037,0,original,0,1.0,0.0,0.0,0.0,0,0,C:\Users\dark_\Downloads\Project_2\Images\1237...
2,1237648705120895059,199.29492,0.527571,0.024114,0,original,0,1.0,0.0,0.0,0.0,0,0,C:\Users\dark_\Downloads\Project_2\Images\1237...
3,1237648720150724863,165.74061,-0.962095,0.033483,0,original,0,1.0,0.0,0.0,0.0,0,0,C:\Users\dark_\Downloads\Project_2\Images\1237...
4,1237649919509594232,31.37202,13.251016,0.024694,0,original,0,1.0,0.0,0.0,0.0,0,0,C:\Users\dark_\Downloads\Project_2\Images\1237...


In [61]:
# ==========================================================
# 5) Portabilidad + Export: rutas reproducibles y guardado CSV
# ==========================================================

from pathlib import Path
import pandas as pd
import numpy as np

def add_file_rel(df: pd.DataFrame, images_dir: Path, col_abs="file_loc", col_rel="file_rel"):
    """
    Crea file_rel relativo a images_dir.
    Ej: C:\...\Images\123.fits  -> Images/123.fits
    """
    df = df.copy()
    images_dir = Path(images_dir).resolve()

    rels = []
    for p in df[col_abs].astype("object").tolist():
        if p is None or (isinstance(p, float) and np.isnan(p)):
            rels.append(np.nan)
            continue
        pth = Path(str(p))
        try:
            rel = pth.resolve().relative_to(images_dir)
            rels.append(str(Path(images_dir.name) / rel).replace("\\", "/"))
        except Exception:
            # fallback: solo el filename
            rels.append(str(Path(images_dir.name) / pth.name).replace("\\", "/"))

    df[col_rel] = rels
    return df

def resolve_file_loc_from_rel(df: pd.DataFrame, images_dir: Path, col_rel="file_rel", col_abs="file_loc"):
    """
    Reconstruye file_loc absoluto desde file_rel y el nuevo images_dir.
    """
    df = df.copy()
    images_dir = Path(images_dir)

    def _to_abs(rel):
        if pd.isna(rel):
            return np.nan
        rel = str(rel).replace("\\", "/")
        # Esperamos "Images/<fname>" o solo "<fname>"
        parts = rel.split("/")
        if len(parts) >= 2 and parts[0].lower() == images_dir.name.lower():
            return str(images_dir / "/".join(parts[1:]))
        else:
            return str(images_dir / parts[-1])

    df[col_abs] = df[col_rel].apply(_to_abs)
    return df

# 1) Crear file_rel (reproducible) a partir de tu IMAGE_DIR actual
df_portable = df_to_dl.copy()
df_portable = add_file_rel(df_portable, IMAGE_DIR, col_abs="file_loc", col_rel="file_rel")

print("Ejemplo de rutas (abs vs rel):")
display(df_portable[["file_loc","file_rel"]].dropna().head(5))

# 2) Guardar CSVs
OUT_DIR = Path("Exports")
OUT_DIR.mkdir(parents=True, exist_ok=True)

csv_all = OUT_DIR / "dataset_all_with_files.csv"
df_portable.to_csv(csv_all, index=False)
print("Guardado:", csv_all.resolve())

# opcional: guardar por dataset también
for dname, part in df_portable.groupby("dataset", dropna=False):
    outp = OUT_DIR / f"dataset_{dname}_with_files.csv"
    part.to_csv(outp, index=False)
    print("Guardado:", outp.resolve())

# 3) Nota para reproducibilidad(ejemplo de uso)
print("\nPara reproducibilidad:")
print("  df = pd.read_csv('dataset_all_with_files.csv')")
print("  NEW_IMAGES_DIR = Path(r'C:\\...\\Images')")
print("  df = resolve_file_loc_from_rel(df, NEW_IMAGES_DIR)")

<>:12: SyntaxWarning: invalid escape sequence '\.'
<>:12: SyntaxWarning: invalid escape sequence '\.'
C:\Users\dark_\AppData\Local\Temp\ipykernel_27344\4158950908.py:12: SyntaxWarning: invalid escape sequence '\.'
  Ej: C:\...\Images\123.fits  -> Images/123.fits


Ejemplo de rutas (abs vs rel):


,file_loc,file_rel
0,C:\Users\dark_\Downloads\Project_2\Images\1237...,Images/1237648721210769659_0.fits
1,C:\Users\dark_\Downloads\Project_2\Images\1237...,Images/1237648705657307354_0.fits
2,C:\Users\dark_\Downloads\Project_2\Images\1237...,Images/1237648705120895059_0.fits
3,C:\Users\dark_\Downloads\Project_2\Images\1237...,Images/1237648720150724863_0.fits
4,C:\Users\dark_\Downloads\Project_2\Images\1237...,Images/1237649919509594232_0.fits


Guardado: C:\Users\dark_\Downloads\Project_2\Exports\dataset_all_with_files.csv
Guardado: C:\Users\dark_\Downloads\Project_2\Exports\dataset_csrg_with_files.csv
Guardado: C:\Users\dark_\Downloads\Project_2\Exports\dataset_manga_with_files.csv
Guardado: C:\Users\dark_\Downloads\Project_2\Exports\dataset_original_with_files.csv

Para reproducibilidad:
  df = pd.read_csv('dataset_all_with_files.csv')
  NEW_IMAGES_DIR = Path(r'C:\...\Images')
  df = resolve_file_loc_from_rel(df, NEW_IMAGES_DIR)


In [9]:
# ==========================================================
# Cargar datasets exportados, reconstruir rutas y guardar CSV corregido
# ==========================================================

import pandas as pd
from pathlib import Path

# Si el notebook está en /notebooks, subir a raíz
PROJECT_ROOT = Path.cwd()

EXPORT_DIR = PROJECT_ROOT / "Exports"
NEW_IMAGES_DIR = PROJECT_ROOT / "Images"   # Project_2 - Copy/Images

# archivos esperados
csv_files = {
    "all": EXPORT_DIR / "dataset_all_with_files_patched.csv",
    "original": EXPORT_DIR / "dataset_original_with_files_patched.csv",
    "manga": EXPORT_DIR / "dataset_manga_with_files_patched.csv",
    "csrg": EXPORT_DIR / "dataset_csrg_with_files_patched.csv",
}

datasets = {}

for name, path in csv_files.items():
    if path.exists():
        df = pd.read_csv(path)
        datasets[name] = df
        print(f"Cargado: {name:8s} -> {df.shape}")
    else:
        print(f"No encontrado: {path}")

if not datasets:
    raise FileNotFoundError(f"No se encontraron CSVs en {EXPORT_DIR}")

# ----------------------------------------------------------
# reconstruir rutas absolutas de imágenes
# ----------------------------------------------------------
def resolve_file_loc_from_rel(df, images_dir):
    df = df.copy()

    def build_path(rel):
        if pd.isna(rel):
            return rel

        rel = str(rel).replace("\\", "/")
        filename = Path(rel).name
        return str((images_dir / filename).resolve())

    df["file_loc"] = df["file_rel"].apply(build_path)
    return df

# ----------------------------------------------------------
# actualizar y guardar cada CSV corregido
# ----------------------------------------------------------
for name, df in datasets.items():
    df_fixed = resolve_file_loc_from_rel(df, NEW_IMAGES_DIR)

    # validación rápida
    exists_count = df_fixed["file_loc"].apply(lambda p: Path(p).exists() if pd.notna(p) else False).sum()
    print(f"\n{name.upper()} -> rutas válidas: {exists_count}/{len(df_fixed)}")

    # guardar sobrescribiendo el archivo original exportado
    out_path = csv_files[name]
    df_fixed.to_csv(out_path, index=False)
    datasets[name] = df_fixed
    print(f"CSV actualizado: {out_path}")

# dataset combinado
if "all" in datasets:
    df_all = datasets["all"]
else:
    df_all = pd.concat(datasets.values(), ignore_index=True)

print(f"\nDataset combinado final: {df_all.shape}")

print("\nEjemplo de rutas reconstruidas:")
display(df_all[["file_rel", "file_loc"]].head())

Cargado: all      -> (10536, 16)
Cargado: original -> (8049, 16)
Cargado: manga    -> (680, 16)
Cargado: csrg     -> (1807, 16)

ALL -> rutas válidas: 10536/10536
CSV actualizado: c:\Users\dark_\Downloads\Project_2 - Copy\Exports\dataset_all_with_files_patched.csv

ORIGINAL -> rutas válidas: 8049/8049
CSV actualizado: c:\Users\dark_\Downloads\Project_2 - Copy\Exports\dataset_original_with_files_patched.csv

MANGA -> rutas válidas: 680/680
CSV actualizado: c:\Users\dark_\Downloads\Project_2 - Copy\Exports\dataset_manga_with_files_patched.csv

CSRG -> rutas válidas: 1807/1807
CSV actualizado: c:\Users\dark_\Downloads\Project_2 - Copy\Exports\dataset_csrg_with_files_patched.csv

Dataset combinado final: (10536, 16)

Ejemplo de rutas reconstruidas:


,file_rel,file_loc
0,Images/1237648721210769659_0.fits,C:\Users\dark_\Downloads\Project_2 - Copy\Imag...
1,Images/1237648705657307354_0.fits,C:\Users\dark_\Downloads\Project_2 - Copy\Imag...
2,Images/1237648705120895059_0.fits,C:\Users\dark_\Downloads\Project_2 - Copy\Imag...
3,Images/1237648720150724863_0.fits,C:\Users\dark_\Downloads\Project_2 - Copy\Imag...
4,Images/1237649919509594232_0.fits,C:\Users\dark_\Downloads\Project_2 - Copy\Imag...
